#  Brazilian E-Commerce (Olist) - Data Understanding & Quality Exploration

##  Mục tiêu
Notebook này thực hiện **khảo sát toàn diện chất lượng dữ liệu (Data Understanding & Data Quality Assessment)** của 9 tập tin CSV thô từ nền tảng thương mại điện tử Olist (`../data/raw/`). 

Đối với từng tập tin, chúng ta sẽ thực hiện phân tích có cấu trúc:
1. **Business Context & Data Dictionary:** Vai trò nghiệp vụ, các khóa chính (PK) và khóa ngoại (FK).
2. **Sample Data:** Trích xuất mẫu dữ liệu đại diện (`head`).
3. **Info & Data Types:** Cấu trúc bảng, kiểu dữ liệu, dung lượng bộ nhớ.
4. **Null / Missing Values Analysis:** Thống kê giá trị khuyết thiếu (số lượng & tỷ lệ %), phân tích nguyên nhân nghiệp vụ.
5. **Duplicate Analysis:** Phân tích trùng lặp toàn bộ dòng và kiểm tra tính duy nhất của Khóa chính.
6. **Descriptive Statistics (Describe):** Phân phối của các biến số (Outliers, Min, Max, Mean) và biến định tính (Top, Frequency).
7. **Cardinality:** Độ phân tán giá trị duy nhất (`nunique`).
8. **Key Takeaways & Cleaning Recommendations:** Đúc kết các vấn đề chất lượng dữ liệu và định hướng xử lý cho bước làm sạch tiếp theo (`cleaning_data.ipynb`).



##  Sơ đồ quan hệ thực thể (ERD - Entity Relationship Diagram)

```mermaid
erDiagram
    CUSTOMERS ||--o{ ORDERS : "places (customer_id)"
    ORDERS ||--|{ ORDER_ITEMS : "contains (order_id)"
    ORDERS ||--|{ PAYMENTS : "paid_by (order_id)"
    ORDERS ||--o{ REVIEWS : "evaluated_by (order_id)"
    SELLERS ||--o{ ORDER_ITEMS : "fulfills (seller_id)"
    PRODUCTS ||--o{ ORDER_ITEMS : "ordered_in (product_id)"
    CATEGORY_TRANSLATION ||--o{ PRODUCTS : "translates (product_category_name)"
    CUSTOMERS }o--|| GEOLOCATION : "located_at (zip_code_prefix)"
    SELLERS }o--|| GEOLOCATION : "located_at (zip_code_prefix)"
```


In [1]:
import os
import sys
import pandas as pd
import numpy as np
import warnings
from IPython.display import display, Markdown, HTML

warnings.filterwarnings('ignore')


pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Libraries and display options successfully initialized!")

Libraries and display options successfully initialized!


## 0. Tải Dữ Liệu (Load Raw Datasets)

Khởi tạo đường dẫn và nạp 9 tập tin CSV vào từ điển `dfs`.


In [2]:
data_dir = "../data/raw"

datasets_info = {
    "customers": {
        "file": "olist_customers_dataset.csv",
        "pk": ["customer_id"],
        "alt_keys": ["customer_unique_id"],
        "desc": "Thông tin định danh và địa lý khách hàng"
    },
    "orders": {
        "file": "olist_orders_dataset.csv",
        "pk": ["order_id"],
        "alt_keys": ["customer_id"],
        "desc": "Bảng trung tâm lưu vòng đời trạng thái và các mốc thời gian đơn hàng"
    },
    "order_items": {
        "file": "olist_order_items_dataset.csv",
        "pk": ["order_id", "order_item_id"],
        "alt_keys": ["product_id", "seller_id"],
        "desc": "Chi tiết từng sản phẩm, giá bán và phí vận chuyển trong mỗi đơn hàng"
    },
    "payments": {
        "file": "olist_order_payments_dataset.csv",
        "pk": ["order_id", "payment_sequential"],
        "alt_keys": None,
        "desc": "Chi tiết phương thức thanh toán, số kỳ trả góp và số tiền thanh toán"
    },
    "reviews": {
        "file": "olist_order_reviews_dataset.csv",
        "pk": ["review_id", "order_id"],
        "alt_keys": ["review_id"],
        "desc": "Điểm số đánh giá (1-5 sao) và phản hồi của khách hàng sau khi nhận hàng"
    },
    "products": {
        "file": "olist_products_dataset.csv",
        "pk": ["product_id"],
        "alt_keys": ["product_category_name"],
        "desc": "Danh mục sản phẩm, độ dài tiêu đề/mô tả và kích thước/trọng lượng hàng hóa"
    },
    "sellers": {
        "file": "olist_sellers_dataset.csv",
        "pk": ["seller_id"],
        "alt_keys": ["seller_zip_code_prefix"],
        "desc": "Thông tin định danh và địa lý các nhà bán lẻ tham gia nền tảng Olist"
    },
    "geolocation": {
        "file": "olist_geolocation_dataset.csv",
        "pk": None,
        "alt_keys": ["geolocation_zip_code_prefix"],
        "desc": "Bản đồ tọa độ kinh độ/vĩ độ theo mã bưu chính (Zip Code Prefix) tại Brazil"
    },
    "category_translation": {
        "file": "product_category_name_translation.csv",
        "pk": ["product_category_name"],
        "alt_keys": None,
        "desc": "Bảng chuyển ngữ danh mục sản phẩm từ tiếng Bồ Đào Nha sang tiếng Anh"
    }
}

dfs = {}
for name, meta in datasets_info.items():
    file_path = os.path.join(data_dir, meta["file"])
    if os.path.exists(file_path):
        dfs[name] = pd.read_csv(file_path)
        print(f"[LOADED] {name:<20}: {dfs[name].shape[0]:>10,d} rows | {dfs[name].shape[1]:>2d} columns")
    else:
        print(f"[ERROR] File not found: {file_path}")

[LOADED] customers           :     99,441 rows |  5 columns
[LOADED] orders              :     99,441 rows |  8 columns
[LOADED] order_items         :    112,650 rows |  7 columns
[LOADED] payments            :    103,886 rows |  5 columns
[LOADED] reviews             :     99,224 rows |  7 columns
[LOADED] products            :     32,951 rows |  9 columns
[LOADED] sellers             :      3,095 rows |  4 columns
[LOADED] geolocation         :  1,000,163 rows |  5 columns
[LOADED] category_translation:         71 rows |  2 columns


## 1. Bảng Tổng Hợp Đối Soát Nhanh (Executive Summary Dashboard)

Bảng dưới đây quét toàn bộ 9 tập tin để so sánh quy mô dòng/cột, dung lượng RAM, tỷ lệ trùng lặp dòng và tỷ lệ khuyết thiếu cao nhất trên mỗi bảng.


In [ ]:
summary_rows = []

for name, meta in datasets_info.items():
    df = dfs[name]
    mem_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
    null_counts = df.isnull().sum()
    null_cols_count = (null_counts > 0).sum()
    max_null_pct = (df.isnull().mean() * 100).max()
    dup_rows = df.duplicated().sum()
    dup_pct = (dup_rows / len(df)) * 100
    
    
    pk_status = "N/A"
    if meta["pk"]:
        pk_dups = df.duplicated(subset=meta["pk"]).sum()
        pk_name = "+".join(meta["pk"])
        pk_status = f"{pk_name} (Dups: {pk_dups})"
    
    summary_rows.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "RAM (MB)": round(mem_mb, 2),
        "Dup Rows": dup_rows,
        "Dup (%)": round(dup_pct, 2),
        "Null Cols": null_cols_count,
        "Max Null (%)": round(max_null_pct, 2),
        "Primary Key Candidate": pk_status
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,Dataset,Rows,Columns,RAM (MB),Dup Rows,Dup (%),Null Cols,Max Null (%),Primary Key Candidate
0,customers,99441,5,26.59,0,0.00,0,0.00,customer_id (Dups: 0)
1,orders,99441,8,52.94,0,0.00,3,2.98,order_id (Dups: 0)
2,order_items,112650,7,35.99,0,0.00,0,0.00,order_id+order_item_id (Dups: 0)
3,payments,103886,5,16.23,0,0.00,0,0.00,order_id+payment_sequential (Dups: 0)
4,reviews,99224,7,39.12,0,0.00,2,88.34,review_id+order_id (Dups: 0)
5,products,32951,9,6.30,0,0.00,8,1.85,product_id (Dups: 0)
6,sellers,3095,4,0.59,0,0.00,0,0.00,seller_id (Dups: 0)
7,geolocation,1000163,5,129.38,261831,26.18,0,0.00,N/A
8,category_translation,71,2,0.01,0,0.00,0,0.00,product_category_name (Dups: 0)


## 2. Hàm Tiện Ích Khảo Sát Chi Tiết Chuẩn Hóa

Hàm `explore_dataset` dưới đây tự động phân tích có hệ thống từng bảng:
- **[1] Mẫu dữ liệu (Sample Data)**
- **[2] Cấu trúc & Kiểu dữ liệu (Info & Dtypes)**
- **[3] Giá trị khuyết thiếu (Null Values)**
- **[4] Dữ liệu trùng lặp (Duplicates & Key Uniqueness)**
- **[5] Thống kê mô tả (Descriptive Statistics: Numerical & Categorical)**
- **[6] Độ đa dạng (Cardinality / Unique Values)**


In [5]:
def explore_dataset(name, df, pk=None, alt_keys=None):
    display(Markdown(f"###  Khảo Sát Chi Tiết Bảng: `{name.upper()}`"))
    
    # [1] Sample Data
    display(Markdown("#### [1] Mẫu Dữ Liệu Đại Diện (5 Dòng Đầu)"))
    display(df.head(5))
    
    # [2] Info & Dtypes
    display(Markdown("#### [2] Cấu Trúc & Kiểu Dữ Liệu (Info & Types)"))
    info_df = pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.astype(str),
        "Non-Null Count": df.notnull().sum().values,
        "Null Count": df.isnull().sum().values,
        "Null (%)": (df.isnull().mean() * 100).round(2).values,
        "Unique Values": df.nunique().values
    })
    display(info_df)
    
    # [3] Null Values Analysis
    display(Markdown("#### [3] Phân Tích Dữ Liệu Khuyết Thiếu (Missing Values)"))
    null_cols = info_df[info_df["Null Count"] > 0].sort_values(by="Null (%)", ascending=False)
    if len(null_cols) > 0:
        display(null_cols[["Column", "Null Count", "Null (%)"]])
    else:
        print("[INFO] Bang nay hoan toan khong co gia tri khuyet (0 Missing Values).")
        
    # [4] Duplicate Analysis
    display(Markdown("#### [4] Phân Tích Dữ Liệu Trùng Lặp (Duplicates)"))
    total_dups = df.duplicated().sum()
    dup_stat = pd.DataFrame({
        "Chỉ số": ["Tổng số dòng", "Số dòng trùng lặp toàn bộ", "Tỷ lệ trùng lặp (%)"],
        "Giá trị": [len(df), total_dups, f"{(total_dups / len(df) * 100):.2f}%"]
    })
    display(dup_stat)
    
    if pk:
        pk_dups = df.duplicated(subset=pk).sum()
        print(f"[PK Check] Khoa chinh {pk}: {pk_dups} dong trung lap.")
    if alt_keys:
        for ak in alt_keys:
            ak_dups = df.duplicated(subset=[ak]).sum()
            print(f"[Key Check] Khoa dinh danh thay the ['{ak}']: {ak_dups} dong lap (So gia tri duy nhat: {df[ak].nunique():,d}).")
            
    # [5] Descriptive Statistics (Describe)
    display(Markdown("#### [5] Thống Kê Mô Tả (Descriptive Statistics)"))
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        display(Markdown("##### • Biến số (Numerical Features):"))
        display(df[num_cols].describe().T)
        
    cat_cols = df.select_dtypes(include=['object']).columns
    if len(cat_cols) > 0:
        display(Markdown("##### • Biến định tính / Chuỗi ký tự (Categorical / String Features):"))
        display(df[cat_cols].describe().T)
        
    print("-" * 100)

---
## 3. Khảo Sát Bảng: `customers` (`olist_customers_dataset.csv`)

###  Nghiệp vụ & Kiến trúc:
- **Vai trò:** Lưu trữ danh mục khách hàng thực hiện các đơn đặt hàng.
- **Khóa chính (PK):** `customer_id` (Mỗi đơn hàng được gán 1 mã `customer_id` duy nhất).
- **Khóa khách hàng thật:** `customer_unique_id` (Định danh thực tế của 1 người mua hàng; 1 người có thể đặt nhiều đơn khác nhau qua nhiều lần mua).
- **Khóa ngoại (FK):** `customer_zip_code_prefix` liên kết với bảng `geolocation`.


In [6]:
explore_dataset(
    name="customers",
    df=dfs["customers"],
    pk=datasets_info["customers"]["pk"],
    alt_keys=datasets_info["customers"]["alt_keys"]
)

###  Khảo Sát Chi Tiết Bảng: `CUSTOMERS`

#### [1] Mẫu Dữ Liệu Đại Diện (5 Dòng Đầu)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


#### [2] Cấu Trúc & Kiểu Dữ Liệu (Info & Types)

,Column,Data Type,Non-Null Count,Null Count,Null (%),Unique Values
customer_id,customer_id,object,99441,0,0.00,99441
customer_unique_id,customer_unique_id,object,99441,0,0.00,96096
customer_zip_code_prefix,customer_zip_code_prefix,int64,99441,0,0.00,14994
customer_city,customer_city,object,99441,0,0.00,4119
customer_state,customer_state,object,99441,0,0.00,27


...

##### • Biến số (Numerical Features):

,count,mean,std,min,25%,50%,75%,max
customer_zip_code_prefix,99441.00,35137.47,29797.94,1003.00,11347.00,24416.00,58900.00,99990.00


##### • Biến định tính / Chuỗi ký tự (Categorical / String Features):

,count,unique,top,freq
customer_id,99441,99441,274fa6071e5e17fe303b9748641082c8,1
customer_unique_id,99441,96096,8d50f5eadf50201ccdcedfb9e2ac8455,17
customer_city,99441,4119,sao paulo,15540
customer_state,99441,27,SP,41746


----------------------------------------------------------------------------------------------------


###  Nhận Xét & Đánh Giá Chất Lượng Dữ Liệu `customers`:
1. **Dữ liệu hoàn thiện:** 100% đầy đủ (99,441 dòng), không có ô nào bị null (`0% missing`).
2. **Không có dòng trùng lặp hoàn toàn:** 0 dòng trùng lặp toàn bộ.
3. **Phân biệt `customer_id` vs `customer_unique_id`:**
   - `customer_id`: 99,441 giá trị duy nhất (độ duy nhất 100%), đóng vai trò làm khóa chính kết nối 1-1 với bảng `orders`.
   - `customer_unique_id`: Có **96,096** khách hàng duy nhất. Khoảng chênh lệch `99,441 - 96,096 = 3,345` đơn hàng đến từ **khách hàng quay lại mua lần 2, lần 3... (Repeat Customers)**. Tỷ lệ khách hàng quay lại là khoảng **3.1%**.
4. **Phân bố địa lý:**
   - Bang có lượng khách hàng lớn nhất là **SP (São Paulo)** với 41,746 khách hàng (~42% toàn quốc), theo sau là RJ (Rio de Janeiro) và MG (Minas Gerais).


---
## 4. Khảo Sát Bảng: `orders` (`olist_orders_dataset.csv`)

###  Nghiệp vụ & Kiến trúc:
- **Vai trò:** Bảng trung tâm cốt lõi của hệ thống, quản lý thông tin trạng thái và các mốc thời gian trong vòng đời xử lý đơn hàng.
- **Khóa chính (PK):** `order_id` (Mỗi đơn hàng là 1 dòng duy nhất).
- **Khóa ngoại (FK):** `customer_id` (liên kết với bảng `customers`).


In [7]:
explore_dataset(
    name="orders",
    df=dfs["orders"],
    pk=datasets_info["orders"]["pk"],
    alt_keys=datasets_info["orders"]["alt_keys"]
)

###  Khảo Sát Chi Tiết Bảng: `ORDERS`

#### [1] Mẫu Dữ Liệu Đại Diện (5 Dòng Đầu)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


#### [2] Cấu Trúc & Kiểu Dữ Liệu (Info & Types)

,Column,Data Type,Non-Null Count,Null Count,Null (%),Unique Values
order_id,order_id,object,99441,0,0.00,99441
customer_id,customer_id,object,99441,0,0.00,99441
order_status,order_status,object,99441,0,0.00,8
order_purchase_timestamp,order_purchase_timestamp,object,99441,0,0.00,98875
order_approved_at,order_approved_at,object,99281,160,0.16,90733
order_delivered_carrier_date,order_delivered_carrier_date,object,97658,1783,1.79,81018
order_delivered_customer_date,order_delivered_customer_date,object,96476,2965,2.98,95664
order_estimated_delivery_date,order_estimated_delivery_date,object,99441,0,0.00,459


...

[PK Check] Khoa chinh ['order_id']: 0 dong trung lap.
[Key Check] Khoa dinh danh thay the ['customer_id']: 0 dong lap (So gia tri duy nhat: 99,441).


#### [5] Thống Kê Mô Tả (Descriptive Statistics)

##### • Biến định tính / Chuỗi ký tự (Categorical / String Features):

,count,unique,top,freq
order_id,99441,99441,66dea50a8b16d9b4dee7af250b4be1a5,1
customer_id,99441,99441,edb027a75a1449115f6b43211ae02a24,1
order_status,99441,8,delivered,96478
order_purchase_timestamp,99441,98875,2018-08-02 12:05:26,3
order_approved_at,99281,90733,2018-02-27 04:31:10,9
order_delivered_carrier_date,97658,81018,2018-05-09 15:48:00,47
order_delivered_customer_date,96476,95664,2018-05-08 19:36:48,3
order_estimated_delivery_date,99441,459,2017-12-20 00:00:00,522


----------------------------------------------------------------------------------------------------


In [8]:
# Phân tích sâu thêm về quan hệ giữa Trạng thái đơn hàng (order_status) và Missing values
display(Markdown("####  Đối soát Trạng thái Đơn hàng vs Dữ liệu khuyết ở các mốc ngày:"))
display(dfs["orders"].groupby("order_status")[["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]].apply(lambda x: x.isnull().sum()))

####  Đối soát Trạng thái Đơn hàng vs Dữ liệu khuyết ở các mốc ngày:

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


###  Nhận Xét & Đánh Giá Chất Lượng Dữ Liệu `orders`:
1. **Tính toàn vẹn khóa chính:** `order_id` hoàn toàn duy nhất (99,441 dòng, 0 duplicates).
2. **Bản chất của các giá trị khuyết (Missing Values):**
   - `order_approved_at`: Thiếu 160 giá trị (chủ yếu ở trạng thái `canceled` hoặc `created`).
   - `order_delivered_carrier_date`: Thiếu 1,783 giá trị (chưa bàn giao cho đơn vị vận chuyển).
   - `order_delivered_customer_date`: Thiếu 2,965 giá trị (2.98%). Bảng đối soát trên chứng minh: **99.7% giá trị thiếu này thuộc về các đơn hàng chưa hoàn tất giao hàng** (`shipped`, `canceled`, `unavailable`, `invoiced`, `processing`). Đây là thiếu sót mang tính logic nghiệp vụ (MNAR - Missing Not At Random), không phải lỗi dữ liệu ngẫu nhiên.
3. **Kiểu dữ liệu cần chuyển đổi:** 5 cột thời gian (`order_purchase_timestamp`, `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`, `order_estimated_delivery_date`) hiện đang ở dạng chuỗi ký tự (`object`). Cần ép kiểu sang `datetime64` ở bước cleaning.


---
## 5. Khảo Sát Bảng: `order_items` (`olist_order_items_dataset.csv`)

###  Nghiệp vụ & Kiến trúc:
- **Vai trò:** Lưu chi tiết từng mặt hàng được mua trong đơn hàng. Một đơn hàng có thể có nhiều mặt hàng (1-N).
- **Khóa chính phức hợp (Composite PK):** `(order_id, order_item_id)`.
- **Khóa ngoại (FK):**
  - `order_id` -> `orders.order_id`
  - `product_id` -> `products.product_id`
  - `seller_id` -> `sellers.seller_id`


In [9]:
explore_dataset(
    name="order_items",
    df=dfs["order_items"],
    pk=datasets_info["order_items"]["pk"],
    alt_keys=datasets_info["order_items"]["alt_keys"]
)

###  Khảo Sát Chi Tiết Bảng: `ORDER_ITEMS`

#### [1] Mẫu Dữ Liệu Đại Diện (5 Dòng Đầu)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


#### [2] Cấu Trúc & Kiểu Dữ Liệu (Info & Types)

,Column,Data Type,Non-Null Count,Null Count,Null (%),Unique Values
order_id,order_id,object,112650,0,0.00,98666
order_item_id,order_item_id,int64,112650,0,0.00,21
product_id,product_id,object,112650,0,0.00,32951
seller_id,seller_id,object,112650,0,0.00,3095
shipping_limit_date,shipping_limit_date,object,112650,0,0.00,93318
price,price,float64,112650,0,0.00,5968
freight_value,freight_value,float64,112650,0,0.00,6999


...

##### • Biến số (Numerical Features):

,count,mean,std,min,25%,50%,75%,max
order_item_id,112650.00,1.20,0.71,1.00,1.00,1.00,1.00,21.00
price,112650.00,120.65,183.63,0.85,39.90,74.99,134.90,6735.00
freight_value,112650.00,19.99,15.81,0.00,13.08,16.26,21.15,409.68


##### • Biến định tính / Chuỗi ký tự (Categorical / String Features):

,count,unique,top,freq
order_id,112650,98666,8272b63d03f5f79c56e9e4120aec44ef,21
product_id,112650,32951,aca2eb7d00ea1a7b8ebd4e68314663af,527
seller_id,112650,3095,6560211a19b47992c3666cc44a7e94c0,2033
shipping_limit_date,112650,93318,2017-07-21 18:25:23,21


----------------------------------------------------------------------------------------------------


###  Nhận Xét & Đánh Giá Chất Lượng Dữ Liệu `order_items`:
1. **Dữ liệu đầy đủ:** 112,650 dòng, 0 missing values trên toàn bộ các cột.
2. **Tính duy nhất của Composite PK:** Cặp `(order_id, order_item_id)` đạt độ duy nhất tuyệt đối (0 duplicates). Có tổng cộng 98,666 đơn hàng xuất hiện trong bảng này (trung bình 1.14 món hàng/đơn).
3. **Phân tích giá bán (`price`):**
   - Giá tối thiểu: **0.85 BRL** | Giá trung vị (50%): **74.99 BRL** | Giá trung bình: **120.65 BRL** | Giá tối đa: **6,735.00 BRL**.
   - Không có mặt hàng nào có giá <= 0. Phân phối lệch phải rõ rệt (Positive Skewness) với một số sản phẩm cao cấp ngoại lai.
4. **Phân tích cước vận chuyển (`freight_value`):**
   - Phí ship tối thiểu là **0.00 BRL** (có 383 mặt hàng được miễn phí vận chuyển - Free Shipping).
   - Phí ship tối đa lên tới **409.68 BRL** (cần lưu ý khi phân tích chi phí giao hàng theo khoảng cách địa lý).


---
## 6. Khảo Sát Bảng: `payments` (`olist_order_payments_dataset.csv`)

###  Nghiệp vụ & Kiến trúc:
- **Vai trò:** Lưu lịch sử các giao dịch thanh toán cho đơn hàng. Một đơn hàng có thể được thanh toán bằng nhiều lần hoặc kết hợp nhiều phương thức (ví dụ: Voucher + Thẻ tín dụng).
- **Khóa chính phức hợp (Composite PK):** `(order_id, payment_sequential)`.
- **Khóa ngoại (FK):** `order_id` -> `orders.order_id`.


In [10]:
explore_dataset(
    name="payments",
    df=dfs["payments"],
    pk=datasets_info["payments"]["pk"],
    alt_keys=datasets_info["payments"]["alt_keys"]
)

###  Khảo Sát Chi Tiết Bảng: `PAYMENTS`

#### [1] Mẫu Dữ Liệu Đại Diện (5 Dòng Đầu)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


#### [2] Cấu Trúc & Kiểu Dữ Liệu (Info & Types)

,Column,Data Type,Non-Null Count,Null Count,Null (%),Unique Values
order_id,order_id,object,103886,0,0.00,99440
payment_sequential,payment_sequential,int64,103886,0,0.00,29
payment_type,payment_type,object,103886,0,0.00,5
payment_installments,payment_installments,int64,103886,0,0.00,24
payment_value,payment_value,float64,103886,0,0.00,29077


...

##### • Biến số (Numerical Features):

,count,mean,std,min,25%,50%,75%,max
payment_sequential,103886.00,1.09,0.71,1.00,1.00,1.00,1.00,29.00
payment_installments,103886.00,2.85,2.69,0.00,1.00,1.00,4.00,24.00
payment_value,103886.00,154.10,217.49,0.00,56.79,100.00,171.84,13664.08


##### • Biến định tính / Chuỗi ký tự (Categorical / String Features):

,count,unique,top,freq
order_id,103886,99440,fa65dad1b0e818e3ccc5cb0e39231352,29
payment_type,103886,5,credit_card,76795


----------------------------------------------------------------------------------------------------


In [12]:
# Kiểm tra chi tiết các phương thức thanh toán và các đơn có giá trị thanh toán = 0
display(Markdown("####  Tần suất các loại hình thanh toán:"))
display(dfs["payments"]["payment_type"].value_counts().to_frame("Count"))

zero_payments = dfs["payments"][dfs["payments"]["payment_value"] == 0]
display(Markdown(f"####  Các giao dịch có `payment_value == 0` (Số lượng: {len(zero_payments)}):"))
display(zero_payments)

####  Tần suất các loại hình thanh toán:

,Count
payment_type,
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529
not_defined,3


####  Các giao dịch có `payment_value == 0` (Số lượng: 9):

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.00


###  Nhận Xét & Đánh Giá Chất Lượng Dữ Liệu `payments`:
1. **Dữ liệu đầy đủ:** 103,886 dòng, 0 missing values.
2. **Khóa chính phức hợp:** Cặp `(order_id, payment_sequential)` đạt độ duy nhất 100% (0 duplicates).
3. **Phổ biến của các phương thức thanh toán:**
   - **Credit Card (Thẻ tín dụng):** Chiếm ưu thế áp đảo với **76,795 giao dịch (~73.9%)**.
   - **Boleto (Hóa đơn ngân hàng):** 19,784 giao dịch (~19.0%) - phương thức rất phổ biến tại Brazil.
   - **Voucher:** 5,775 giao dịch (~5.6%).
   - **Debit Card:** 1,529 giao dịch (~1.5%).
   - **`not_defined`:** Có đúng **3 dòng** có loại thanh toán không xác định. Cần lọc hoặc xử lý ở bước cleaning.
4. **Giao dịch giá trị 0:** Có **9 giao dịch** có `payment_value == 0` (tất cả đều có số tiền thanh toán = 0, cần kiểm tra đối chiếu khi tính doanh thu).
5. **Số kỳ trả góp (`payment_installments`):** Dao động từ 0 đến 24 tháng. Trung vị là 1 lần, nhưng có các giao dịch trả góp tối đa 24 tháng. Có 2 dòng trả góp 0 kỳ cần xử lý.


---
## 7. Khảo Sát Bảng: `reviews` (`olist_order_reviews_dataset.csv`)

###  Nghiệp vụ & Kiến trúc:
- **Vai trò:** Lưu trữ đánh giá của khách hàng sau khi nhận hàng hoặc trải nghiệm dịch vụ.
- **Khóa chính:** Cặp `(review_id, order_id)` (Lưu ý: 1 `review_id` có thể được liên kết với nhiều `order_id` nếu người mua cùng lúc nhiều đơn).
- **Khóa ngoại (FK):** `order_id` -> `orders.order_id`.


In [13]:
explore_dataset(
    name="reviews",
    df=dfs["reviews"],
    pk=datasets_info["reviews"]["pk"],
    alt_keys=datasets_info["reviews"]["alt_keys"]
)

###  Khảo Sát Chi Tiết Bảng: `REVIEWS`

#### [1] Mẫu Dữ Liệu Đại Diện (5 Dòng Đầu)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


#### [2] Cấu Trúc & Kiểu Dữ Liệu (Info & Types)

,Column,Data Type,Non-Null Count,Null Count,Null (%),Unique Values
review_id,review_id,object,99224,0,0.00,98410
order_id,order_id,object,99224,0,0.00,98673
review_score,review_score,int64,99224,0,0.00,5
review_comment_title,review_comment_title,object,11568,87656,88.34,4527
review_comment_message,review_comment_message,object,40977,58247,58.70,36159
review_creation_date,review_creation_date,object,99224,0,0.00,636
review_answer_timestamp,review_answer_timestamp,object,99224,0,0.00,98248


...

##### • Biến số (Numerical Features):

,count,mean,std,min,25%,50%,75%,max
review_score,99224.00,4.09,1.35,1.00,4.00,5.00,5.00,5.00


##### • Biến định tính / Chuỗi ký tự (Categorical / String Features):

,count,unique,top,freq
review_id,99224,98410,08528f70f579f0c830189efc523d2182,3
order_id,99224,98673,df56136b8031ecd28e200bb18e6ddb2e,3
review_comment_title,11568,4527,Recomendo,423
review_comment_message,40977,36159,Muito bom,230
review_creation_date,99224,636,2017-12-19 00:00:00,463
review_answer_timestamp,99224,98248,2017-06-15 23:21:05,4


----------------------------------------------------------------------------------------------------


In [14]:
# Phân bố điểm đánh giá (review_score)
score_dist = dfs["reviews"]["review_score"].value_counts().sort_index().to_frame("Số lượng")
score_dist["Tỷ lệ (%)"] = (score_dist["Số lượng"] / len(dfs["reviews"]) * 100).round(2)
display(Markdown("####  Phân phối điểm đánh giá sao (1 -> 5 sao):"))
display(score_dist)

####  Phân phối điểm đánh giá sao (1 -> 5 sao):

,Số lượng,Tỷ lệ (%)
review_score,,
1,11424,11.51
2,3151,3.18
3,8179,8.24
4,19142,19.29
5,57328,57.78


###  Nhận Xét & Đánh Giá Chất Lượng Dữ Liệu `reviews`:
1. **Dữ liệu khuyết thiếu tập trung ở văn bản nhận xét:**
   - `review_comment_title`: Khuyết **87,656 dòng (88.34%)**. Phần lớn khách hàng không đặt tiêu đề.
   - `review_comment_message`: Khuyết **58,247 dòng (58.70%)**. Hơn một nửa khách hàng chỉ bấm chấm sao mà không viết nội dung chi tiết.
   - Chiến lược cleaning: Điền nhãn `"No Title"` và `"No Message"` để không làm mất bản ghi điểm số khi phân tích.
2. **Khóa chính & Trùng lặp:**
   - `review_id` đơn lẻ có **814 bản ghi bị trùng** (do 1 phản hồi được hệ thống gắn cho nhiều đơn hàng cùng giỏ).
   - Tuy nhiên, cặp `(review_id, order_id)` đạt độ duy nhất 100% (0 duplicates).
3. **Phân phối điểm số (Sentiment):**
   - Đánh giá **5 sao chiếm tới 57.78%**, 4 sao chiếm 19.29% (Tổng mức độ hài lòng tích cực đạt > 77%).
   - Tỷ lệ 1 sao chiếm **11.51%** (chủ yếu liên quan đến giao hàng chậm trễ hoặc hàng không đúng mô tả).


---
## 8. Khảo Sát Bảng: `products` (`olist_products_dataset.csv`)

###  Nghiệp vụ & Kiến trúc:
- **Vai trò:** Danh mục thông tin kỹ thuật của các sản phẩm được bày bán trên Olist.
- **Khóa chính (PK):** `product_id`.
- **Khóa ngoại (FK):** `product_category_name` -> `product_category_name_translation.csv`.


In [15]:
explore_dataset(
    name="products",
    df=dfs["products"],
    pk=datasets_info["products"]["pk"],
    alt_keys=datasets_info["products"]["alt_keys"]
)

###  Khảo Sát Chi Tiết Bảng: `PRODUCTS`

#### [1] Mẫu Dữ Liệu Đại Diện (5 Dòng Đầu)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1.00,225.00,16.00,10.00,14.00
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.00,276.00,1.00,1000.00,30.00,18.00,20.00
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.00,250.00,1.00,154.00,18.00,9.00,15.00
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.00,261.00,1.00,371.00,26.00,4.00,26.00
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.00,402.00,4.00,625.00,20.00,17.00,13.00


#### [2] Cấu Trúc & Kiểu Dữ Liệu (Info & Types)

,Column,Data Type,Non-Null Count,Null Count,Null (%),Unique Values
product_id,product_id,object,32951,0,0.00,32951
product_category_name,product_category_name,object,32341,610,1.85,73
product_name_lenght,product_name_lenght,float64,32341,610,1.85,66
product_description_lenght,product_description_lenght,float64,32341,610,1.85,2960
product_photos_qty,product_photos_qty,float64,32341,610,1.85,19
product_weight_g,product_weight_g,float64,32949,2,0.01,2204
product_length_cm,product_length_cm,float64,32949,2,0.01,99
product_height_cm,product_height_cm,float64,32949,2,0.01,102
product_width_cm,product_width_cm,float64,32949,2,0.01,95


...

##### • Biến số (Numerical Features):

,count,mean,std,min,25%,50%,75%,max
product_name_lenght,32341.00,48.48,10.25,5.00,42.00,51.00,57.00,76.00
product_description_lenght,32341.00,771.50,635.12,4.00,339.00,595.00,972.00,3992.00
product_photos_qty,32341.00,2.19,1.74,1.00,1.00,1.00,3.00,20.00
product_weight_g,32949.00,2276.47,4282.04,0.00,300.00,700.00,1900.00,40425.00
product_length_cm,32949.00,30.82,16.91,7.00,18.00,25.00,38.00,105.00
product_height_cm,32949.00,16.94,13.64,2.00,8.00,13.00,21.00,105.00
product_width_cm,32949.00,23.20,12.08,6.00,15.00,20.00,30.00,118.00


##### • Biến định tính / Chuỗi ký tự (Categorical / String Features):

,count,unique,top,freq
product_id,32951,32951,106392145fca363410d287a815be6de4,1
product_category_name,32341,73,cama_mesa_banho,3029


----------------------------------------------------------------------------------------------------


In [16]:
# Kiểm tra các sản phẩm bị thiếu danh mục và kích thước
missing_prods = dfs["products"][dfs["products"]["product_category_name"].isnull()]
display(Markdown(f"####  Số sản phẩm bị khuyết danh mục: {len(missing_prods)} dòng. Xem mẫu:"))
display(missing_prods.head(5))

####  Số sản phẩm bị khuyết danh mục: 610 dòng. Xem mẫu:

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.00,17.00,14.00,12.00
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.00,16.00,7.00,20.00
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.00,20.00,20.00,20.00
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.00,41.00,30.00,41.00
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.00,35.00,7.00,12.00


###  Nhận Xét & Đánh Giá Chất Lượng Dữ Liệu `products`:
1. **Tính duy nhất:** `product_id` hoàn toàn duy nhất (32,951 sản phẩm, 0 duplicates).
2. **Cụm dữ liệu khuyết đồng thời (Block Missing):**
   - Đúng **610 sản phẩm (1.85%)** bị thiếu đồng thời: `product_category_name`, `product_name_lenght`, `product_description_lenght`, và `product_photos_qty`.
   - Có **2 sản phẩm** bị thiếu các thông số kích thước/trọng lượng (`product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm`).
   - Chiến lược cleaning: Có thể điền danh mục khuyết thành `"outro"` hoặc `"unknown"` để bảo toàn doanh số của các sản phẩm này khi join với `order_items`.
3. **Kích thước & Trọng lượng:**
   - Trọng lượng dao động từ 0g (hoặc vài chục gram) đến 40.42 kg.
   - Chiều dài, rộng, cao dao động từ vài cm đến hơn 1 mét.


---
## 9. Khảo Sát Bảng: `sellers` (`olist_sellers_dataset.csv`)

###  Nghiệp vụ & Kiến trúc:
- **Vai trò:** Danh bạ thông tin định danh và khu vực hoạt động của các nhà bán hàng (Merchants).
- **Khóa chính (PK):** `seller_id`.
- **Khóa ngoại (FK):** `seller_zip_code_prefix` -> `geolocation.geolocation_zip_code_prefix`.


In [17]:
explore_dataset(
    name="sellers",
    df=dfs["sellers"],
    pk=datasets_info["sellers"]["pk"],
    alt_keys=datasets_info["sellers"]["alt_keys"]
)

###  Khảo Sát Chi Tiết Bảng: `SELLERS`

#### [1] Mẫu Dữ Liệu Đại Diện (5 Dòng Đầu)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


#### [2] Cấu Trúc & Kiểu Dữ Liệu (Info & Types)

,Column,Data Type,Non-Null Count,Null Count,Null (%),Unique Values
seller_id,seller_id,object,3095,0,0.00,3095
seller_zip_code_prefix,seller_zip_code_prefix,int64,3095,0,0.00,2246
seller_city,seller_city,object,3095,0,0.00,611
seller_state,seller_state,object,3095,0,0.00,23


...

##### • Biến số (Numerical Features):

,count,mean,std,min,25%,50%,75%,max
seller_zip_code_prefix,3095.00,32291.06,32713.45,1001.00,7093.50,14940.00,64552.50,99730.00


##### • Biến định tính / Chuỗi ký tự (Categorical / String Features):

,count,unique,top,freq
seller_id,3095,3095,9e25199f6ef7e7c347120ff175652c3b,1
seller_city,3095,611,sao paulo,694
seller_state,3095,23,SP,1849


----------------------------------------------------------------------------------------------------


###  Nhận Xét & Đánh Giá Chất Lượng Dữ Liệu `sellers`:
1. **Dữ liệu hoàn thiện:** 3,095 người bán, 0 missing values, 0 dòng trùng lặp.
2. **Tính duy nhất của khóa chính:** `seller_id` duy nhất 100%.
3. **Tập trung thị trường người bán:**
   - Bang **SP (São Paulo)** chiếm tới **1,849 người bán (~59.7%)**, cho thấy São Paulo là trung tâm logistics và bán hàng cốt lõi của nền tảng Olist.
   - Các bang tiếp theo là PR (Paraná, 349 sellers) và MG (Minas Gerais, 244 sellers).


---
## 10. Khảo Sát Bảng: `geolocation` (`olist_geolocation_dataset.csv`)

###  Nghiệp vụ & Kiến trúc:
- **Vai trò:** Bảng tra cứu tọa độ địa lý (Vĩ độ `lat`, Kinh độ `lng`, Thành phố, Bang) dựa trên tiền tố mã bưu điện 5 chữ số (`geolocation_zip_code_prefix`).
- **Quan hệ:** Được tham chiếu từ `customers.customer_zip_code_prefix` và `sellers.seller_zip_code_prefix`.


In [18]:
explore_dataset(
    name="geolocation",
    df=dfs["geolocation"],
    pk=datasets_info["geolocation"]["pk"],
    alt_keys=datasets_info["geolocation"]["alt_keys"]
)

###  Khảo Sát Chi Tiết Bảng: `GEOLOCATION`

#### [1] Mẫu Dữ Liệu Đại Diện (5 Dòng Đầu)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.55,-46.64,sao paulo,SP
1,1046,-23.55,-46.64,sao paulo,SP
2,1046,-23.55,-46.64,sao paulo,SP
3,1041,-23.54,-46.64,sao paulo,SP
4,1035,-23.54,-46.64,sao paulo,SP


#### [2] Cấu Trúc & Kiểu Dữ Liệu (Info & Types)

,Column,Data Type,Non-Null Count,Null Count,Null (%),Unique Values
geolocation_zip_code_prefix,geolocation_zip_code_prefix,int64,1000163,0,0.00,19015
geolocation_lat,geolocation_lat,float64,1000163,0,0.00,717360
geolocation_lng,geolocation_lng,float64,1000163,0,0.00,717613
geolocation_city,geolocation_city,object,1000163,0,0.00,8011
geolocation_state,geolocation_state,object,1000163,0,0.00,27


...

##### • Biến số (Numerical Features):

,count,mean,std,min,25%,50%,75%,max
geolocation_zip_code_prefix,1000163.00,36574.17,30549.34,1001.00,11075.00,26530.00,63504.00,99990.00
geolocation_lat,1000163.00,-21.18,5.72,-36.61,-23.60,-22.92,-19.98,45.07
geolocation_lng,1000163.00,-46.39,4.27,-101.47,-48.57,-46.64,-43.77,121.11


##### • Biến định tính / Chuỗi ký tự (Categorical / String Features):

,count,unique,top,freq
geolocation_city,1000163,8011,sao paulo,135800
geolocation_state,1000163,27,SP,404268


----------------------------------------------------------------------------------------------------


In [19]:
# Phân tích sâu về trùng lặp và giá trị tọa độ ngoại lai ngoài lãnh thổ Brazil
display(Markdown("####  Kiểm tra tọa độ ngoại lai ngoài lãnh thổ Brazil (Vĩ độ [-34, +5.5], Kinh độ [-74, -34]):"))
outliers_lat = dfs["geolocation"][(dfs["geolocation"]["geolocation_lat"] > 5.5) | (dfs["geolocation"]["geolocation_lat"] < -34)]
outliers_lng = dfs["geolocation"][(dfs["geolocation"]["geolocation_lng"] > -34) | (dfs["geolocation"]["geolocation_lng"] < -74)]
print(f"So ban ghi co vi do (Lat) ngoai Brazil: {len(outliers_lat):,d}")
print(f"So ban ghi co kinh do (Lng) ngoai Brazil: {len(outliers_lng):,d}")

####  Kiểm tra tọa độ ngoại lai ngoài lãnh thổ Brazil (Vĩ độ [-34, +5.5], Kinh độ [-74, -34]):

So ban ghi co vi do (Lat) ngoai Brazil: 31
So ban ghi co kinh do (Lng) ngoai Brazil: 37


###  Nhận Xét & Đánh Giá Chất Lượng Dữ Liệu `geolocation`:
1. **Quy mô lớn nhất:** 1,000,163 dòng dữ liệu, 0 missing values.
2. **Vấn đề trùng lặp nghiêm trọng nhất:**
   - Có tới **261,831 dòng trùng lặp hoàn toàn 100% (26.18%)**!
   - 1 mã zip code prefix có rất nhiều dòng ghi nhận tọa độ khác nhau (chỉ có **19,015 mã zip code duy nhất** trên hơn 1 triệu dòng).
   - **Chiến lược xử lý cho bước làm sạch (`cleaning_data.ipynb`):**
     1. Loại bỏ các dòng trùng lặp (`drop_duplicates()`).
     2. Lọc bỏ các tọa độ ngoại lai ngoài lãnh thổ Brazil (như lat > 5.5 hoặc lng > -34).
     3. Nhóm theo `geolocation_zip_code_prefix` và tính tọa độ trung bình (`mean()`) hoặc lấy dòng xuất hiện nhiều nhất để tạo bảng tra cứu 1-1 tinh gọn (19,015 dòng) trước khi nạp vào PostgreSQL.


---
## 11. Khảo Sát Bảng: `category_translation` (`product_category_name_translation.csv`)

###  Nghiệp vụ & Kiến trúc:
- **Vai trò:** Bảng đối chiếu từ điển dịch tên danh mục sản phẩm từ tiếng Bồ Đào Nha sang tiếng Anh.
- **Khóa chính (PK):** `product_category_name`.


In [20]:
explore_dataset(
    name="category_translation",
    df=dfs["category_translation"],
    pk=datasets_info["category_translation"]["pk"],
    alt_keys=datasets_info["category_translation"]["alt_keys"]
)

###  Khảo Sát Chi Tiết Bảng: `CATEGORY_TRANSLATION`

#### [1] Mẫu Dữ Liệu Đại Diện (5 Dòng Đầu)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


#### [2] Cấu Trúc & Kiểu Dữ Liệu (Info & Types)

,Column,Data Type,Non-Null Count,Null Count,Null (%),Unique Values
product_category_name,product_category_name,object,71,0,0.00,71
product_category_name_english,product_category_name_english,object,71,0,0.00,71


...

[PK Check] Khoa chinh ['product_category_name']: 0 dong trung lap.


#### [5] Thống Kê Mô Tả (Descriptive Statistics)

##### • Biến định tính / Chuỗi ký tự (Categorical / String Features):

,count,unique,top,freq
product_category_name,71,71,beleza_saude,1
product_category_name_english,71,71,health_beauty,1


----------------------------------------------------------------------------------------------------


In [21]:
# Kiểm tra xem có danh mục nào trong products mà CHƯA có trong bảng dịch hay không
prod_cats = set(dfs["products"]["product_category_name"].dropna().unique())
trans_cats = set(dfs["category_translation"]["product_category_name"].unique())
missing_translations = prod_cats - trans_cats

display(Markdown(f"####  Các danh mục có trong bảng `products` nhưng CHƯA có bản dịch ({len(missing_translations)} danh mục):"))
display(list(missing_translations))

####  Các danh mục có trong bảng `products` nhưng CHƯA có bản dịch (2 danh mục):

['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']

###  Nhận Xét & Đánh Giá Chất Lượng Dữ Liệu `category_translation`:
1. **Dữ liệu sạch:** 71 dòng, 0 missing values, 0 trùng lặp, khóa chính duy nhất 100%.
2. **Phát hiện quan trọng (Data Leakage / Incomplete Translation):**
   - Bảng `products` có tổng cộng 73 danh mục độc lập.
   - Có **2 danh mục** xuất hiện trong `products` nhưng **không có** trong bảng `category_translation`:
     - `'pc_gamer'` -> Dịch tiếng Anh: `'pc_gamer'`
     - `'portateis_cozinha_e_preparadores_de_alimentos'` -> Dịch tiếng Anh: `'small_appliances_kitchen_and_food_preparers'`
   - **Hệ quả & Giải pháp:** Nếu dùng `INNER JOIN` khi ghép bảng, các sản phẩm thuộc 2 danh mục này sẽ bị mất khỏi báo cáo. Cần thực hiện `LEFT JOIN` và bổ sung bản dịch thủ công cho 2 danh mục này ở bước cleaning.


---
## 12. Tổng Kết & Bản Đồ Chiến Lược Làm Sạch Dữ Liệu (Data Cleaning Roadmap)

Dưới đây là bảng tổng hợp các vấn đề cốt lõi được phát hiện qua quá trình khám phá dữ liệu và giải pháp cụ thể sẽ được áp dụng trong notebook `cleaning_data.ipynb`:

| Bảng dữ liệu | Vấn đề phát hiện | Giải pháp đề xuất tại `cleaning_data.ipynb` |
| :--- | :--- | :--- |
| **`orders`** | • 5 cột ngày tháng đang lưu dạng string (`object`).<br>• 2,965 đơn thiếu ngày giao khách (hầu hết do chưa giao). | • Ép kiểu sang `datetime64[ns]`.<br>• Giữ nguyên `NaT` cho ngày chưa giao, có thể tạo thêm feature `delivery_status_flag`. |
| **`order_items`** | • Cột `shipping_limit_date` là string.<br>• Phí ship có 383 mặt hàng = 0. | • Ép kiểu sang `datetime64[ns]`.<br>• Giữ nguyên giá trị 0 (đây là chương trình Free Ship hợp lệ). |
| **`payments`** | • 3 dòng có `payment_type = 'not_defined'`.<br>• 9 dòng có `payment_value = 0.0`. | • Loại bỏ hoặc quy về `unknown`.<br>• Kiểm tra và lọc các dòng thanh toán 0 đồng nếu không có voucher hỗ trợ. |
| **`reviews`** | • 88.34% thiếu title, 58.70% thiếu message.<br>• 814 `review_id` bị lặp (nhưng cặp `review_id` + `order_id` là duy nhất). | • Điền `review_comment_title = 'No Title'`.<br>• Điền `review_comment_message = 'No Message'`.<br>• Giữ nguyên khóa phức hợp `(review_id, order_id)`. |
| **`products`** | • 610 sản phẩm thiếu danh mục và kích thước.<br>• 2 sản phẩm thiếu trọng lượng. | • Điền danh mục khuyết thành `'unknown'`.<br>• Impute trọng lượng/kích thước bằng trung vị (`median`) theo danh mục tương ứng. |
| **`geolocation`** | • 261,831 dòng trùng lặp hoàn toàn (26.18%).<br>• Tồn tại tọa độ ngoại lai ngoài lãnh thổ Brazil.<br>• Hơn 1 triệu dòng gây nặng nề cho database. | • Bỏ trùng lặp (`drop_duplicates`).<br>• Lọc bỏ tọa độ ngoại lai ngoài Brazil.<br>• Nhóm theo `zip_code_prefix` và lấy tọa độ trung bình, thu gọn về ~19,015 dòng chuẩn. |
| **`category_translation`** | • Thiếu 2 danh mục: `pc_gamer` và `portateis_cozinha_e_preparadores_de_alimentos`. | • Thêm trực tiếp 2 bản dịch này vào bảng trước khi ánh xạ sang tiếng Anh. |
